# Run on Pi5 terminal

In [ ]:
pip install firebase-admin fluvio ultralytics opencv-python

# code for pi5_stream.py

In [ ]:
import cv2
import time
import numpy as np
from ultralytics import YOLO
from fluvio import Fluvio
import firebase_admin
from firebase_admin import credentials, db

# ==========================================
#  CONFIGURATION
# ==========================================
# 1. FIREBASE SETUP
# Paste your Realtime Database URL here (from the Firebase Console)
FIREBASE_DB_URL = 'https://crowdsense-detection-default-rtdb.firebaseio.com/' 

# 2. FLUVIO SETUP
TOPIC_NAME = "crowd-stream"  

# 3. TUNING PARAMETERS
FALL_CONFIRMATION_TIME = 1.5  # Seconds a person must be down to trigger alert
CRITICAL_BUFFER_TIME = 3.0    # Seconds crowd must be high to trigger stampede alert
CROWD_THRESHOLD = 15          # Number of people to trigger "High Density"

# ==========================================
#  SYSTEM INITIALIZATION
# ==========================================
print("🔥 Connecting to Firebase Database...")
try:
    cred = credentials.Certificate("firebase_key.json") 
    firebase_admin.initialize_app(cred, {
        'databaseURL': FIREBASE_DB_URL
    })
    db_ref = db.reference('crowd_monitor/zone_A')
    print("✅ Firebase Connected!")
except Exception as e:
    print(f"❌ Firebase Error: {e}")
    exit(1)

print(f" Connecting to Fluvio Topic: {TOPIC_NAME}...")
try:
    fluvio = Fluvio.connect()
    producer = fluvio.topic_producer(TOPIC_NAME)
    print("✅ Fluvio Connected!")
except Exception as e:
    print(f"⚠️ Fluvio Error: {e} (Video will not stream, but AI will work)")
    producer = None

print(" Loading YOLOv11 Nano Model...")
model = YOLO('yolo11n.pt') 

# Initialize Camera
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

# ==========================================
# GLOBAL VARIABLES
# ==========================================
start_critical_time = None   # Timer for Stampede
start_fall_time = None       # Timer for Fall Confirmation
last_firebase_update = 0     # Throttle for DB writes
last_fall_alert = 0          # Throttle for Phone Calls

print("SYSTEM LIVE: Monitoring Crowd & Falls...")

while True:
    ret, frame = cap.read()
    if not ret: 
        print("❌ Camera Error")
        break

    # 1. RUN AI DETECTION
    # We use a lower confidence (0.15) to catch people lying down
    results = model.predict(frame, conf=0.15, iou=0.80, imgsz=320, classes=[0], verbose=False)
    
    total_persons = len(results[0].boxes) if results[0].boxes else 0
    status_text = "Normal"
    box_color = (0, 255, 0) # Green by default

    # ==========================================
    # LOGIC 1: FALL DETECTION (Time-Based)
    # ==========================================
    is_anyone_falling_now = False

    if results[0].boxes:
        for box in results[0].boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            w = x2 - x1
            h = y2 - y1
            
            # GEOMETRY CHECK: If Width > 1.2x Height (Horizontal)
            if w > 1.2 * h:
                is_anyone_falling_now = True
                # Draw Amber Box (Potential Fall)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 165, 255), 2)
                cv2.putText(frame, "Checking...", (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,165,255), 1)

    # PERSISTENCE CHECK (The Anti-Flicker Logic)
    if is_anyone_falling_now:
        if start_fall_time is None:
            start_fall_time = time.time() # Start Timer
        
        elif time.time() - start_fall_time > FALL_CONFIRMATION_TIME:
            #  FALL CONFIRMED (Down for > 1.5s)
            status_text = "FALL DETECTED"
            box_color = (0, 0, 255) # Red
            
            # Visuals
            cv2.putText(frame, "!!! FALL CONFIRMED !!!", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 3)
            
            # UPDATE FIREBASE (Triggers Laptop Alarm)
            # 30s Cooldown to prevent spamming
            if time.time() - last_fall_alert > 30:
                print("🚨 FALL ALERT SENT TO CLOUD!")
                db_ref.update({
                    'status': "FALL DETECTED",
                    'fall_trigger': True,
                    'last_fall_time': time.time()
                })
                last_fall_alert = time.time()
    else:
        # If they stood up or disappeared, RESET timer immediately
        start_fall_time = None

    # ==========================================
    #  LOGIC 2: CROWD DENSITY (Buffer)
    # ==========================================
    # Only run this if we didn't already find a fall
    if status_text != "FALL DETECTED":
        if total_persons > CROWD_THRESHOLD:
            if start_critical_time is None:
                start_critical_time = time.time()
            
            elapsed = time.time() - start_critical_time
            if elapsed > CRITICAL_BUFFER_TIME:
                status_text = "CRITICAL RISK"
                box_color = (0, 0, 255) # Red
            else:
                status_text = "High Density"
                box_color = (0, 165, 255) # Orange
        else:
            start_critical_time = None # Reset
            status_text = "Normal"

    # ==========================================
    #  DATA SYNC (Firebase)
    # ==========================================
    # Send updates every 0.5s to keep dashboard live
    current_time = time.time()
    if current_time - last_firebase_update > 0.5:
        try:
            db_ref.update({
                'count': total_persons,
                'status': status_text,
                'timestamp': current_time
            })
            last_firebase_update = current_time
        except Exception:
            pass # Ignore temporary network blips

    # ==========================================
    #  VIDEO STREAM (Fluvio)
    # ==========================================
    # Stream ALWAYS if there is a fall, or if crowd > 5 (for demo purposes)
    if producer:
        should_stream = True # Set to True for Demo so you always see video
        
        if should_stream:
            # Draw standard boxes for the video feed
            if results[0].boxes:
                for box in results[0].boxes:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    # Use the color determined by logic above
                    cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)

            # Compress image to JPEG (Quality 40 to save Wifi speed)
            ret, buffer = cv2.imencode('.jpg', frame, [int(cv2.IMWRITE_JPEG_QUALITY), 40])
            if ret:
                producer.send_record(buffer.tobytes(), 0)

cap.release()

# Code for Gateway.py

In [ ]:
from flask import Flask, Response
from fluvio import Fluvio
from flask_cors import CORS

app = Flask(__name__)
CORS(app)  # Allows your React app (on a different device) to talk to this

# CONFIGURATION
TOPIC_NAME = "crowd-stream"  # Important Must match what is in pi_stream.py

# Connect to Fluvio (Running locally on the Pi)
try:
    fluvio = Fluvio.connect()
    # We are gonna consume from the "end" so we get live video, not old history
    consumer = fluvio.partition_consumer(TOPIC_NAME, 0)
    print("✅ Gateway Connected to Fluvio! Ready to serve video.")
except Exception as e:
    print(f"❌ Fluvio Connection Error: {e}")
    consumer = None

def generate_frames():
    if not consumer:
        return
        
    # Stream frames from Fluvio to the Web app interface we made
    for record in consumer.stream(0):
        yield (b'--frame\r\n'
               b'Content-Type: image/jpeg\r\n\r\n' + record.value() + b'\r\n')

@app.route('/video_feed')
def video_feed():
    return Response(generate_frames(), mimetype='multipart/x-mixed-replace; boundary=frame')

if __name__ == '__main__':
    # Host='0.0.0.0' makes it accessible to your Laptop/Mobile React App
    app.run(host='0.0.0.0', port=5000, threaded=True)